In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, balanced_accuracy_score, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import ConfusionMatrixDisplay

# Datos

https://archive.ics.uci.edu/dataset/186/wine+quality

In [ ]:
# Cargar datasets de vino tinto y blanco
url_red = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
url_white = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv"

df_red = pd.read_csv(url_red, sep=';')
df_white = pd.read_csv(url_white, sep=';')

# Agregar columna indicando el tipo de vino
df_red['wine_type'] = 'red'
df_white['wine_type'] = 'white'

# Unir ambos datasets
df = pd.concat([df_red, df_white], axis=0).reset_index(drop=True)

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
# Graficar histogramas 
df.hist(figsize=(12, 8), bins=20, edgecolor='black')
plt.tight_layout()
plt.show()

In [ ]:
# Graficar boxplots 
df.plot(kind='box', figsize=(12, 8), vert=True, subplots=True, layout=(4, 4), sharex=False, sharey=False)
plt.tight_layout()
plt.show()

In [ ]:
df.groupby('quality').size()/df.shape[0]

In [ ]:
df = df[df['quality'].isin([5, 6, 7])].reset_index(drop=True)

# Tratamiento de los datos

In [ ]:
df_cat=df.loc[:,['wine_type']]
df_num=df.drop(columns=['wine_type','quality'])

scaler = StandardScaler()
df_num_scale = scaler.fit_transform(df_num)
df_num_scale= pd.DataFrame(df_num_scale,columns=df_num.columns.values)

df_cat=df_cat.reset_index(drop=True);df_num_scale=df_num_scale.reset_index(drop=True)
df_final=pd.concat([df_cat,df_num_scale],axis=1)

df_final = pd.get_dummies(df_final,drop_first=True)
df_final.head()

# Modelo de clasificacion

In [ ]:
# Dividir datos en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(df_final,df['quality'],random_state = 123,test_size=0.2, stratify=df['quality'])

In [ ]:
# Definir hiperparámetros a optimizar
param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [10, 20, 30, None],
    'max_features': ['sqrt', 'log2'],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False]
}

In [ ]:
# Definir modelo base
rf = RandomForestClassifier(random_state=42)

# Aplicar búsqueda aleatoria con Cross Validation
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=20,  # Número de combinaciones a probar
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),  # Cross Validation con 5 folds
    scoring='balanced_accuracy',
    n_jobs=-1,
    random_state=42
)

# Entrenar modelo con búsqueda de hiperparámetros
random_search.fit(X_train, y_train)

In [ ]:
# Mostrar los mejores parámetros encontrados
best_params = random_search.best_params_
print(f"Mejores hiperparámetros: {best_params}")

In [ ]:
# Evaluar el modelo optimizado en el conjunto de prueba
best_rf = random_search.best_estimator_
y_pred = best_rf.predict(X_test)

In [ ]:
print("=== REPORTE DE CLASIFICACIÓN ===")
print(classification_report(y_test, y_pred, zero_division=0))

# Graficar la Matriz de Confusión
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_estimator(
    best_rf, 
    X_test, 
    y_test, 
    cmap='Blues', 
    ax=ax, 
    colorbar=False
)
plt.title('Matriz de Confusión - Random Forest', fontsize=14)
plt.show()

In [ ]:
importancias = best_rf.feature_importances_
columnas = X_train.columns

df_importancias = pd.DataFrame({'Variable': columnas, 'Importancia': importancias})
df_importancias = df_importancias.sort_values(by='Importancia', ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(df_importancias['Variable'], df_importancias['Importancia'], color='mediumseagreen', edgecolor='black')
plt.title('Importancia de las Variables Predictoras', fontsize=14)
plt.xlabel('Importancia Relativa (Gini)')
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.show()